In [1]:
import ee
import pandas as pd

ee.Initialize(project="alien-cedar-483108-h3")
print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [2]:
import ee
import pandas as pd

# Aurad_3 coordinates
lat = 18.07666667
lon = 76.91916667

point = ee.Geometry.Point([lon, lat])

era5 = (
    ee.ImageCollection("ECMWF/ERA5/HOURLY")
    .filterDate("2023-07-15", "2023-07-18")
    .filterBounds(point)
)

print("Number of ERA5 hourly images:", era5.size().getInfo())

Number of ERA5 hourly images: 72


In [3]:
def get_era5_value(image):
    values = image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=31000
    )
    
    return ee.Feature(None, {
        "time": image.date().format("YYYY-MM-dd HH:mm"),
        "temperature_K": values.get("temperature_2m")
    })

era5_features = era5.map(get_era5_value)

era5_data = era5_features.getInfo()

era5_df = pd.DataFrame([
    feature["properties"]
    for feature in era5_data["features"]
])

era5_df["temperature_C"] = era5_df["temperature_K"] - 273.15

era5_df.head()

,temperature_K,time,temperature_C
0,297.789307,2023-07-15 00:00,24.639307
1,297.729828,2023-07-15 01:00,24.579828
2,298.375885,2023-07-15 02:00,25.225885
3,298.194031,2023-07-15 03:00,25.044031
4,298.287079,2023-07-15 04:00,25.137079


In [4]:
era5_df["time"] = pd.to_datetime(era5_df["time"], utc=True)
era5_df["time_ist"] = era5_df["time"].dt.tz_convert("Asia/Kolkata")

era5_df.head()

,temperature_K,time,temperature_C,time_ist
0,297.789307,2023-07-15 00:00:00+00:00,24.639307,2023-07-15 05:30:00+05:30
1,297.729828,2023-07-15 01:00:00+00:00,24.579828,2023-07-15 06:30:00+05:30
2,298.375885,2023-07-15 02:00:00+00:00,25.225885,2023-07-15 07:30:00+05:30
3,298.194031,2023-07-15 03:00:00+00:00,25.044031,2023-07-15 08:30:00+05:30
4,298.287079,2023-07-15 04:00:00+00:00,25.137079,2023-07-15 09:30:00+05:30


In [5]:
station_df = pd.read_csv("../data/raw/maha_temp_raw.csv")

aurad3 = station_df[station_df["Station"] == "Aurad_3"].copy()

aurad3["time"] = pd.to_datetime(
    aurad3["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M"
)

aurad3[["time", "Air Temperature Telemetry Hourly (AoC)"]].head(10)

,time,Air Temperature Telemetry Hourly (AoC)
2743678,2025-12-21 22:00:00,14.1
2743679,2023-07-15 11:15:00,27.5
2743680,2023-07-15 14:00:00,28.2
2743681,2023-07-15 15:00:00,27.6
2743682,2023-07-15 18:00:00,27.6
2743683,2023-07-15 19:00:00,26.1
2743684,2023-07-15 20:00:00,23.7
2743685,2023-07-15 21:00:00,23.7
2743686,2023-07-15 22:00:00,23.8
2743687,2023-07-16 00:00:00,23.9


In [6]:
aurad3["hour"] = aurad3["time"].dt.floor("h")

hourly_station = (
    aurad3.groupby("hour")["Air Temperature Telemetry Hourly (AoC)"]
    .mean()
    .reset_index()
)

hourly_station.head(10)

,hour,Air Temperature Telemetry Hourly (AoC)
0,2023-07-15 11:00:00,27.5
1,2023-07-15 14:00:00,28.2
2,2023-07-15 15:00:00,27.6
3,2023-07-15 18:00:00,27.6
4,2023-07-15 19:00:00,26.1
5,2023-07-15 20:00:00,23.7
6,2023-07-15 21:00:00,23.7
7,2023-07-15 22:00:00,23.8
8,2023-07-16 00:00:00,23.9
9,2023-07-16 03:00:00,23.3


In [7]:
hourly_station["hour"] = (
    pd.to_datetime(hourly_station["hour"])
    .dt.tz_localize("Asia/Kolkata")
)

hourly_station.head()

,hour,Air Temperature Telemetry Hourly (AoC)
0,2023-07-15 11:00:00+05:30,27.5
1,2023-07-15 14:00:00+05:30,28.2
2,2023-07-15 15:00:00+05:30,27.6
3,2023-07-15 18:00:00+05:30,27.6
4,2023-07-15 19:00:00+05:30,26.1


In [8]:
era5_df["hour"] = era5_df["time_ist"].dt.floor("h")

comparison = pd.merge(
    hourly_station,
    era5_df[["hour", "temperature_C"]],
    on="hour",
    how="inner"
)

comparison.head(10)

,hour,Air Temperature Telemetry Hourly (AoC),temperature_C
0,2023-07-15 11:00:00+05:30,27.5,28.710291
1,2023-07-15 14:00:00+05:30,28.2,27.847314
2,2023-07-15 15:00:00+05:30,27.6,27.566888
3,2023-07-15 18:00:00+05:30,27.6,26.721460
4,2023-07-15 19:00:00+05:30,26.1,26.796899
5,2023-07-15 20:00:00+05:30,23.7,25.168176
6,2023-07-15 21:00:00+05:30,23.7,25.043115
7,2023-07-15 22:00:00+05:30,23.8,24.295648
8,2023-07-16 00:00:00+05:30,23.9,24.010858
9,2023-07-16 03:00:00+05:30,23.3,23.033105


In [9]:
print("Aurad_3 hourly observations:", len(hourly_station))
print("Matched ERA5 observations:", len(comparison))

Aurad_3 hourly observations: 18425
Matched ERA5 observations: 11


In [10]:
comparison[["hour", "Air Temperature Telemetry Hourly (AoC)", "temperature_C"]]

,hour,Air Temperature Telemetry Hourly (AoC),temperature_C
0,2023-07-15 11:00:00+05:30,27.5,28.710291
1,2023-07-15 14:00:00+05:30,28.2,27.847314
2,2023-07-15 15:00:00+05:30,27.6,27.566888
3,2023-07-15 18:00:00+05:30,27.6,26.721460
4,2023-07-15 19:00:00+05:30,26.1,26.796899
5,2023-07-15 20:00:00+05:30,23.7,25.168176
6,2023-07-15 21:00:00+05:30,23.7,25.043115
7,2023-07-15 22:00:00+05:30,23.8,24.295648
8,2023-07-16 00:00:00+05:30,23.9,24.010858
9,2023-07-16 03:00:00+05:30,23.3,23.033105


In [11]:
era5_elevation = (
    ee.ImageCollection("ECMWF/ERA5/HOURLY")
    .filterDate("2023-07-15", "2023-07-16")
    .first()
    .select("geopotential")
)

elev_value = era5_elevation.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=31000
).getInfo()

elev_value

{'geopotential': 5821.265625}

In [12]:
era5_elevation_m = elev_value["geopotential"] / 9.80665

print(f"ERA5 elevation: {era5_elevation_m:.2f} m")

ERA5 elevation: 593.60 m


In [13]:
srtm = ee.Image("USGS/SRTMGL1_003")

srtm_value = srtm.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30
).getInfo()

srtm_value

{'elevation': 565}

In [14]:
elevation_difference = 565 - era5_elevation_m
temperature_correction = -0.0065 * elevation_difference

print(f"Elevation difference: {elevation_difference:.2f} m")
print(f"Temperature correction: {temperature_correction:.3f} °C")

Elevation difference: -28.60 m
Temperature correction: 0.186 °C


In [15]:
comparison["physical_prediction_C"] = (
    comparison["temperature_C"] + temperature_correction
)

comparison["error_C"] = (
    comparison["physical_prediction_C"]
    - comparison["Air Temperature Telemetry Hourly (AoC)"]
)

comparison[[
    "hour",
    "Air Temperature Telemetry Hourly (AoC)",
    "temperature_C",
    "physical_prediction_C",
    "error_C"
]]

,hour,Air Temperature Telemetry Hourly (AoC),temperature_C,physical_prediction_C,error_C
0,2023-07-15 11:00:00+05:30,27.5,28.710291,28.896216,1.396216
1,2023-07-15 14:00:00+05:30,28.2,27.847314,28.033240,-0.166760
2,2023-07-15 15:00:00+05:30,27.6,27.566888,27.752814,0.152814
3,2023-07-15 18:00:00+05:30,27.6,26.721460,26.907385,-0.692615
4,2023-07-15 19:00:00+05:30,26.1,26.796899,26.982825,0.882825
5,2023-07-15 20:00:00+05:30,23.7,25.168176,25.354102,1.654102
6,2023-07-15 21:00:00+05:30,23.7,25.043115,25.229041,1.529041
7,2023-07-15 22:00:00+05:30,23.8,24.295648,24.481574,0.681574
8,2023-07-16 00:00:00+05:30,23.9,24.010858,24.196783,0.296783
9,2023-07-16 03:00:00+05:30,23.3,23.033105,23.219031,-0.080969


In [16]:
mae = comparison["error_C"].abs().mean()
rmse = (comparison["error_C"] ** 2).mean() ** 0.5
bias = comparison["error_C"].mean()

print(f"MAE:  {mae:.3f} °C")
print(f"RMSE: {rmse:.3f} °C")
print(f"Bias: {bias:.3f} °C")

MAE:  0.721 °C
RMSE: 0.907 °C
Bias: 0.550 °C


In [17]:
station_counts = (
    station_df.groupby("Station")
    .size()
    .sort_values(ascending=False)
)

station_counts.head(20)

Station
Sarud                   79019
Barhanpur               76273
Nazare_1                74548
Umbre (Kasurdi)         74402
Rahimatpur              68592
Nighoje                 67119
Shivade                 66694
Bhigwan                 65998
Patryachiwadi           64669
Rosa (Sina Kolegaon)    63454
Gudhe                   63046
Khadakwasala_2          60928
Urmodi(Parali)          54910
Bhuinj                  54367
Chandgad_1              52682
Ambale_1                47890
Sakhar                  46373
Songe_Bange             42081
Takli Barur             41595
Suksale                 34443
dtype: int64

Scaking observation analysis for 5 stations

In [18]:
test_stations = [
    "Sarud",
    "Rahimatpur",
    "Chandgad_1",
    "Rosa (Sina Kolegaon)",
    "Aurad_3"
]

print(test_stations)

['Sarud', 'Rahimatpur', 'Chandgad_1', 'Rosa (Sina Kolegaon)', 'Aurad_3']


In [19]:
station_info = (
    station_df[station_df["Station"].isin(test_stations)]
    .groupby("Station")
    .agg(
        latitude=("Latitude", "first"),
        longitude=("Longitude", "first"),
        start=("Data Acquisition Time", "min"),
        end=("Data Acquisition Time", "max"),
        observations=("Station", "size")
    )
    .reset_index()
)

station_info

,Station,latitude,longitude,start,end,observations
0,Aurad_3,18.076667,76.919167,01-01-2024 00:00,31-12-2025 23:00,23298
1,Chandgad_1,15.936667,74.176944,01-01-2023 00:00,31-12-2022 23:45,52682
2,Rahimatpur,17.593333,74.198889,01-01-2024 00:00,31-12-2023 23:45,68592
3,Rosa (Sina Kolegaon),18.492778,75.738611,01-01-2024 00:30,31-12-2023 23:45,63454
4,Sarud,16.897778,74.043333,01-01-2024 00:00,31-12-2023 23:45,79019


In [20]:
station_df["datetime"] = pd.to_datetime(
    station_df["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M"
)

station_info = (
    station_df[station_df["Station"].isin(test_stations)]
    .groupby("Station")
    .agg(
        latitude=("Latitude", "first"),
        longitude=("Longitude", "first"),
        start=("datetime", "min"),
        end=("datetime", "max"),
        observations=("Station", "size")
    )
    .reset_index()
)

station_info

,Station,latitude,longitude,start,end,observations
0,Aurad_3,18.076667,76.919167,2023-07-15 11:15:00,2025-12-31 23:00:00,23298
1,Chandgad_1,15.936667,74.176944,2022-02-12 14:00:00,2024-12-18 07:15:00,52682
2,Rahimatpur,17.593333,74.198889,2021-05-22 14:30:00,2024-12-18 17:00:00,68592
3,Rosa (Sina Kolegaon),18.492778,75.738611,2022-02-12 14:00:00,2024-11-23 08:30:00,63454
4,Sarud,16.897778,74.043333,2022-02-12 14:15:00,2024-12-18 17:00:00,79019


In [21]:
test_points = [
    ("Aurad_3", 18.076667, 76.919167),
    ("Chandgad_1", 15.936667, 74.176944),
    ("Rahimatpur", 17.593333, 74.198889),
    ("Rosa (Sina Kolegaon)", 18.492778, 75.738611),
    ("Sarud", 16.897778, 74.043333),
]

srtm = ee.Image("USGS/SRTMGL1_003")

features = []

for name, lat, lon in test_points:
    point = ee.Geometry.Point([lon, lat])

    value = srtm.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30
    ).getInfo()

    features.append({
        "Station": name,
        "SRTM_elevation_m": value["elevation"]
    })

srtm_station_df = pd.DataFrame(features)

srtm_station_df

,Station,SRTM_elevation_m
0,Aurad_3,565
1,Chandgad_1,716
2,Rahimatpur,651
3,Rosa (Sina Kolegaon),729
4,Sarud,554


In [22]:
era5_collection = (
    ee.ImageCollection("ECMWF/ERA5/HOURLY")
    .filterDate("2023-07-15", "2023-07-16")
)

era5_first = era5_collection.first().select("geopotential")

era5_elevations = []

for name, lat, lon in test_points:
    point = ee.Geometry.Point([lon, lat])

    value = era5_first.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=31000
    ).getInfo()

    geopotential = value["geopotential"]
    elevation_m = geopotential / 9.80665

    era5_elevations.append({
        "Station": name,
        "ERA5_elevation_m": elevation_m
    })

era5_station_df = pd.DataFrame(era5_elevations)

era5_station_df

,Station,ERA5_elevation_m
0,Aurad_3,593.603894
1,Chandgad_1,725.639617
2,Rahimatpur,723.899925
3,Rosa (Sina Kolegaon),634.075275
4,Sarud,647.993007


In [23]:
elevation_comparison = pd.merge(
    srtm_station_df,
    era5_station_df,
    on="Station"
)

elevation_comparison["elevation_difference_m"] = (
    elevation_comparison["SRTM_elevation_m"]
    - elevation_comparison["ERA5_elevation_m"]
)

elevation_comparison["temperature_correction_C"] = (
    -0.0065 * elevation_comparison["elevation_difference_m"]
)

elevation_comparison

,Station,SRTM_elevation_m,ERA5_elevation_m,elevation_difference_m,temperature_correction_C
0,Aurad_3,565,593.603894,-28.603894,0.185925
1,Chandgad_1,716,725.639617,-9.639617,0.062658
2,Rahimatpur,651,723.899925,-72.899925,0.473850
3,Rosa (Sina Kolegaon),729,634.075275,94.924725,-0.617011
4,Sarud,554,647.993007,-93.993007,0.610955


In [24]:
era5_collection = (
    ee.ImageCollection("ECMWF/ERA5/HOURLY")
    .filterDate("2023-07-15", "2023-07-22")
)

print("ERA5 hourly images:", era5_collection.size().getInfo())

ERA5 hourly images: 168


In [25]:
test_stations = {
    "Aurad_3": (18.07666667, 76.91916667),
    "Sarud": (16.89777778, 74.04333333),
    "Rahimatpur": (17.59333333, 74.19888889),
    "Chandgad_1": (15.93666667, 74.17694444),
    "Rosa (Sina Kolegaon)": (18.49277778, 75.73861111),
}

print(test_stations)

{'Aurad_3': (18.07666667, 76.91916667), 'Sarud': (16.89777778, 74.04333333), 'Rahimatpur': (17.59333333, 74.19888889), 'Chandgad_1': (15.93666667, 74.17694444), 'Rosa (Sina Kolegaon)': (18.49277778, 75.73861111)}


In [26]:
def extract_station_era5(station_name, coords):
    lat, lon = coords
    point = ee.Geometry.Point([lon, lat])

    def extract_image(image):
        values = image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=31000
        )

        return ee.Feature(None, {
            "station": station_name,
            "time": image.date().format("YYYY-MM-dd HH:mm"),
            "temperature_K": values.get("temperature_2m"),
            "geopotential": values.get("geopotential")
        })

    features = era5_collection.map(extract_image)
    result = features.getInfo()

    return pd.DataFrame([
        feature["properties"]
        for feature in result["features"]
    ])


era5_station_data = []

for station_name, coords in test_stations.items():
    df = extract_station_era5(station_name, coords)
    era5_station_data.append(df)

era5_all = pd.concat(era5_station_data, ignore_index=True)

era5_all["temperature_C"] = era5_all["temperature_K"] - 273.15
era5_all["time"] = pd.to_datetime(era5_all["time"], utc=True)
era5_all["time_ist"] = era5_all["time"].dt.tz_convert("Asia/Kolkata")

print("Total rows:", len(era5_all))
print(era5_all.head())

Total rows: 840
   geopotential  station  temperature_K                      time  \
0   5821.265625  Aurad_3     297.789307 2023-07-15 00:00:00+00:00   
1   5821.265625  Aurad_3     297.729828 2023-07-15 01:00:00+00:00   
2   5821.265625  Aurad_3     298.375885 2023-07-15 02:00:00+00:00   
3   5821.265625  Aurad_3     298.194031 2023-07-15 03:00:00+00:00   
4   5821.265625  Aurad_3     298.287079 2023-07-15 04:00:00+00:00   

   temperature_C                  time_ist  
0      24.639307 2023-07-15 05:30:00+05:30  
1      24.579828 2023-07-15 06:30:00+05:30  
2      25.225885 2023-07-15 07:30:00+05:30  
3      25.044031 2023-07-15 08:30:00+05:30  
4      25.137079 2023-07-15 09:30:00+05:30  


In [27]:
station_test = station_df[
    station_df["Station"].isin(test_stations.keys())
].copy()

station_test["time"] = pd.to_datetime(
    station_test["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M"
)

# Convert observations to hourly values
station_test["hour"] = station_test["time"].dt.floor("h")

hourly_station_all = (
    station_test
    .groupby(["Station", "hour"])["Air Temperature Telemetry Hourly (AoC)"]
    .mean()
    .reset_index()
)

# Treat station timestamps as IST
hourly_station_all["hour"] = (
    hourly_station_all["hour"]
    .dt.tz_localize("Asia/Kolkata")
)

print("Hourly station observations:", len(hourly_station_all))
print(hourly_station_all.head())

Hourly station observations: 73364
   Station                      hour  Air Temperature Telemetry Hourly (AoC)
0  Aurad_3 2023-07-15 11:00:00+05:30                                    27.5
1  Aurad_3 2023-07-15 14:00:00+05:30                                    28.2
2  Aurad_3 2023-07-15 15:00:00+05:30                                    27.6
3  Aurad_3 2023-07-15 18:00:00+05:30                                    27.6
4  Aurad_3 2023-07-15 19:00:00+05:30                                    26.1


In [29]:
era5_all["hour"] = (
    era5_all["time"]
    .dt.floor("h")
)

# Convert UTC hour to IST
era5_all["hour"] = era5_all["hour"].dt.tz_convert("Asia/Kolkata")

print(era5_all[["station", "time", "hour"]].head())

   station                      time                      hour
0  Aurad_3 2023-07-15 00:00:00+00:00 2023-07-15 05:30:00+05:30
1  Aurad_3 2023-07-15 01:00:00+00:00 2023-07-15 06:30:00+05:30
2  Aurad_3 2023-07-15 02:00:00+00:00 2023-07-15 07:30:00+05:30
3  Aurad_3 2023-07-15 03:00:00+00:00 2023-07-15 08:30:00+05:30
4  Aurad_3 2023-07-15 04:00:00+00:00 2023-07-15 09:30:00+05:30


In [30]:
print(hourly_station_all[["Station", "hour"]].head())
print(era5_all[["station", "hour"]].head())

   Station                      hour
0  Aurad_3 2023-07-15 11:00:00+05:30
1  Aurad_3 2023-07-15 14:00:00+05:30
2  Aurad_3 2023-07-15 15:00:00+05:30
3  Aurad_3 2023-07-15 18:00:00+05:30
4  Aurad_3 2023-07-15 19:00:00+05:30
   station                      hour
0  Aurad_3 2023-07-15 05:30:00+05:30
1  Aurad_3 2023-07-15 06:30:00+05:30
2  Aurad_3 2023-07-15 07:30:00+05:30
3  Aurad_3 2023-07-15 08:30:00+05:30
4  Aurad_3 2023-07-15 09:30:00+05:30


In [31]:
hourly_station_all["hour_utc"] = (
    hourly_station_all["hour"]
    .dt.tz_convert("UTC")
)

print(hourly_station_all[["Station", "hour", "hour_utc"]].head())

   Station                      hour                  hour_utc
0  Aurad_3 2023-07-15 11:00:00+05:30 2023-07-15 05:30:00+00:00
1  Aurad_3 2023-07-15 14:00:00+05:30 2023-07-15 08:30:00+00:00
2  Aurad_3 2023-07-15 15:00:00+05:30 2023-07-15 09:30:00+00:00
3  Aurad_3 2023-07-15 18:00:00+05:30 2023-07-15 12:30:00+00:00
4  Aurad_3 2023-07-15 19:00:00+05:30 2023-07-15 13:30:00+00:00


In [32]:
hourly_station_all["hour_utc"] = (
    hourly_station_all["hour"]
    .dt.tz_convert("UTC")
    .dt.round("h")
)

print(hourly_station_all[["Station", "hour", "hour_utc"]].head())

   Station                      hour                  hour_utc
0  Aurad_3 2023-07-15 11:00:00+05:30 2023-07-15 06:00:00+00:00
1  Aurad_3 2023-07-15 14:00:00+05:30 2023-07-15 08:00:00+00:00
2  Aurad_3 2023-07-15 15:00:00+05:30 2023-07-15 10:00:00+00:00
3  Aurad_3 2023-07-15 18:00:00+05:30 2023-07-15 12:00:00+00:00
4  Aurad_3 2023-07-15 19:00:00+05:30 2023-07-15 14:00:00+00:00


In [33]:
comparison_all = pd.merge(
    hourly_station_all,
    era5_all[["station", "time", "temperature_C", "geopotential"]],
    left_on=["Station", "hour_utc"],
    right_on=["station", "time"],
    how="inner"
)

print("Matched observations:", len(comparison_all))
print(comparison_all.head())

Matched observations: 460
   Station                      hour  Air Temperature Telemetry Hourly (AoC)  \
0  Aurad_3 2023-07-15 11:00:00+05:30                                    27.5   
1  Aurad_3 2023-07-15 14:00:00+05:30                                    28.2   
2  Aurad_3 2023-07-15 15:00:00+05:30                                    27.6   
3  Aurad_3 2023-07-15 18:00:00+05:30                                    27.6   
4  Aurad_3 2023-07-15 19:00:00+05:30                                    26.1   

                   hour_utc  station                      time  temperature_C  \
0 2023-07-15 06:00:00+00:00  Aurad_3 2023-07-15 06:00:00+00:00      28.710291   
1 2023-07-15 08:00:00+00:00  Aurad_3 2023-07-15 08:00:00+00:00      28.791315   
2 2023-07-15 10:00:00+00:00  Aurad_3 2023-07-15 10:00:00+00:00      27.566888   
3 2023-07-15 12:00:00+00:00  Aurad_3 2023-07-15 12:00:00+00:00      26.573328   
4 2023-07-15 14:00:00+00:00  Aurad_3 2023-07-15 14:00:00+00:00      26.796899   

   geo

In [34]:
srtm_elevations = {
    "Aurad_3": 565,
    "Chandgad_1": 716,
    "Rahimatpur": 651,
    "Rosa (Sina Kolegaon)": 729,
    "Sarud": 554
}

# Convert ERA5 geopotential to approximate elevation in metres
comparison_all["era5_elevation_m"] = (
    comparison_all["geopotential"] / 9.80665
)

# Add SRTM elevation
comparison_all["srtm_elevation_m"] = (
    comparison_all["Station"].map(srtm_elevations)
)

# Elevation difference: local terrain - ERA5 terrain
comparison_all["delta_elevation_m"] = (
    comparison_all["srtm_elevation_m"]
    - comparison_all["era5_elevation_m"]
)

# Temperature correction using 6.5°C/km
comparison_all["temperature_correction_C"] = (
    -0.0065 * comparison_all["delta_elevation_m"]
)

# Final physics-based prediction
comparison_all["physical_prediction_C"] = (
    comparison_all["temperature_C"]
    + comparison_all["temperature_correction_C"]
)

# Prediction error
comparison_all["error_C"] = (
    comparison_all["physical_prediction_C"]
    - comparison_all["Air Temperature Telemetry Hourly (AoC)"]
)

print(comparison_all[
    [
        "Station",
        "temperature_C",
        "srtm_elevation_m",
        "era5_elevation_m",
        "temperature_correction_C",
        "physical_prediction_C",
        "Air Temperature Telemetry Hourly (AoC)",
        "error_C"
    ]
].head(10))

   Station  temperature_C  srtm_elevation_m  era5_elevation_m  \
0  Aurad_3      28.710291               565        593.603894   
1  Aurad_3      28.791315               565        593.603894   
2  Aurad_3      27.566888               565        593.603894   
3  Aurad_3      26.573328               565        593.603894   
4  Aurad_3      26.796899               565        593.603894   
5  Aurad_3      26.796899               565        593.603894   
6  Aurad_3      25.043115               565        593.603894   
7  Aurad_3      25.043115               565        593.603894   
8  Aurad_3      24.019006               565        593.603894   
9  Aurad_3      23.033105               565        593.603894   

   temperature_correction_C  physical_prediction_C  \
0                  0.185925              28.896216   
1                  0.185925              28.977240   
2                  0.185925              27.752814   
3                  0.185925              26.759253   
4             

In [35]:
station_metrics = []

for station, group in comparison_all.groupby("Station"):
    errors = group["error_C"]

    mae = errors.abs().mean()
    rmse = (errors ** 2).mean() ** 0.5
    bias = errors.mean()

    station_metrics.append({
        "Station": station,
        "Matched_hours": len(group),
        "MAE_C": mae,
        "RMSE_C": rmse,
        "Bias_C": bias
    })

metrics_df = pd.DataFrame(station_metrics)

print(metrics_df.to_string(index=False))

             Station  Matched_hours    MAE_C   RMSE_C    Bias_C
             Aurad_3             11 1.006414 1.330580  0.838829
          Chandgad_1            124 0.599042 0.776827  0.479587
Rosa (Sina Kolegaon)            157 1.720222 2.113369 -1.703210
               Sarud            168 0.713762 0.870534  0.332042
